# M³Diff — thử LoRA + SCST (4 ô)

**Chạy 1 → 2 → 3 → 4**, dùng tài khoản Drive cũ và **A100 80GB**. Không cần chạy lại notebook trước.

- **G — gốc** để biết có cải thiện hay không. **A — LoRA thường** để tách tác dụng của SCST. **B — LoRA + SCST** học thêm từ điểm CIDEr của câu tự sinh.
- A/B cùng 512 cặp train, mỗi bản 128 cập nhật. Đây là pilot mới, chưa có bằng chứng tăng điểm. Ô 3 đo tốc độ trước; trần **180 phút cộng dồn từ lúc chuẩn bị nạp model**, không phải thời gian hứa chạy xong. Cài thư viện/tải ảnh ở ô 1–2 nằm ngoài trần này; Colab vẫn tính tài nguyên khi GPU còn kết nối.
- Tự lưu Drive mỗi **5 cập nhật** và từng caption. Mất phiên: mở lại file, chạy **1 → 2 → 3**, tự tiếp tục từ lần lưu gần nhất trong ngân sách còn lại. Có thể làm lại tối đa 5 cập nhật nếu ngắt đúng lúc đang lưu; không tự xóa thư mục hay đổi cấu hình.
- Chấm 200 cặp development, tách Real/Render. Chỉ mở thêm 200 cặp confirmation không trùng ảnh nếu B vượt ngưỡng đặt trước. Validation từng được dùng trong dự án; kết quả này chưa thay full test.
- Dừng, lỗi hoặc không đủ ngân sách: vẫn chạy **ô 4**, gửi `M3Diff_LoRA_SCST_bao_cao.zip`. Chỉ lấy báo cáo ở phiên mới: dùng CPU, chạy **1 → 4**. Ô 3 hoàn tất sẽ giải phóng model GPU nhưng bạn vẫn cần ngắt runtime khi xong.

Không đổi demo, không dùng checkpoint MDP trộn đã thất bại, không tự thử tiếp cấu hình khác.


In [ ]:
#@title 1. Kết nối Drive và tạo môi trường — chạy một lần mỗi phiên
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import base64,gzip,hashlib,importlib.util,json,os,shutil,signal,subprocess,sys,threading,time,zipfile
STORAGE=Path('/content/drive/MyDrive/M3Diff_Storage')
if not STORAGE.is_dir():raise FileNotFoundError('Không thấy M3Diff_Storage trong MyDrive. Dùng đúng tài khoản Drive của lần trước.')
RUN_ID='scst_lora_20261031'
OUT=STORAGE/'MDP_Projector_v1/experiments'/RUN_ID
OUT.mkdir(parents=True,exist_ok=True)
WORK=Path('/content/m3diff_scst_tools');WORK.mkdir(exist_ok=True)
SOURCE=Path('/content/m3diff_scst_source');SOURCE.mkdir(exist_ok=True)
VENV=Path('/content/m3diff-train-venv');PY=VENV/'bin/python'
EMBEDDED_SOURCES = {'scst_support.py': '"""CPU-only data, provenance, caption cache and predeclared pilot gate."""\nimport hashlib,json,math,os,random,shutil\nfrom collections import Counter,defaultdict\nfrom pathlib import Path,PurePosixPath\n\nSEED=20261031\nREVISION=\'ed4a84cc51912c73e3d54c8643256d9469657552\'\nQUESTION=\'What modifications can be observed between these two pictures?  <image><image>Difference:\'\nMETRICS=(\'BLEU-4\',\'METEOR\',\'ROUGE-L\',\'CIDEr\')\nARMS=(\'G\',\'A\',\'B\')\nTRAIN_CONFIG={\'seed\':SEED,\'pairs\':512,\'updates\':128,\'accumulation\':4,\'lr\':1e-5,\n    \'weight_decay\':0.,\'clip_grad\':1.,\'lora_r\':16,\'lora_alpha\':32,\'lora_dropout\':0.,\n    \'scst_weight\':0.05,\'checkpoint_every\':5,\'budget_seconds\':10800,\'max_new_tokens\':256}\n\ndef should_stop_before_model(eta,budget,completed=False):\n    return completed or (eta is not None and eta.get(\'admitted\') is False) or (budget is not None and budget[\'limit\']-budget[\'spent\']<600)\n\ndef read(p):return json.loads(Path(p).read_text(encoding=\'utf-8\'))\ndef identity(x):return hashlib.sha256(json.dumps(x,sort_keys=True,ensure_ascii=False).encode()).hexdigest()\ndef sha(p):\n    h=hashlib.sha256()\n    with Path(p).open(\'rb\') as f:\n        for b in iter(lambda:f.read(8*1024*1024),b\'\'):h.update(b)\n    return h.hexdigest()\ndef atomic(p,x):\n    p=Path(p);p.parent.mkdir(parents=True,exist_ok=True);tmp=p.with_name(p.name+\'.tmp\')\n    tmp.write_text(json.dumps(x,ensure_ascii=False,indent=2)+\'\\n\',encoding=\'utf-8\');os.replace(tmp,p)\ndef require_identity(p,x):\n    if Path(p).exists():\n        if read(p)!=x:raise ValueError(\'Cấu hình khác lần đang chạy: \'+str(p))\n    else:atomic(p,x)\ndef safe_relative(s):\n    p=PurePosixPath(s)\n    if not s or p.is_absolute() or \'..\' in p.parts or \'\\\\\' in s or \':\' in s:raise ValueError(\'Unsafe path\')\n    return p\ndef copy_checked(src,dst,expected):\n    dst=Path(dst);dst.parent.mkdir(parents=True,exist_ok=True)\n    if dst.is_file() and sha(dst)==expected:return\n    tmp=dst.with_name(dst.name+\'.incomplete\')\n    with Path(src).open(\'rb\') as a,tmp.open(\'wb\') as b:shutil.copyfileobj(a,b,8*1024*1024)\n    if sha(tmp)!=expected:raise ValueError(\'Copy checksum mismatch: \'+str(src))\n    os.replace(tmp,dst)\ndef verify_hub_digest(p,d):\n    if d[\'kind\']==\'sha256\':return sha(p)==d[\'value\']\n    if d[\'kind\']!=\'git-sha1\':raise ValueError(\'Unknown Hub digest\')\n    p=Path(p);h=hashlib.sha1((\'blob \'+str(p.stat().st_size)+\'\\0\').encode())\n    with p.open(\'rb\') as f:\n        for b in iter(lambda:f.read(8*1024*1024),b\'\'):h.update(b)\n    return h.hexdigest()==d[\'value\']\n\ndef groups(rows):\n    parent={}\n    def root(x):\n        parent.setdefault(x,x)\n        if parent[x]!=x:parent[x]=root(parent[x])\n        return parent[x]\n    for r in rows:\n        a,b=map(root,r[\'image\']);parent[max(a,b)]=min(a,b)\n    result=defaultdict(list)\n    for r in rows:result[root(r[\'image\'][0])].append(r)\n    return dict(sorted(result.items()))\n\ndef spread(group_lists,count,rng):\n    pools=[sorted(v,key=lambda r:r[\'id\'])[:] for v in group_lists]\n    for pool in pools:rng.shuffle(pool)\n    result=[];i=0\n    while len(result)<count:\n        before=len(result)\n        for pool in pools:\n            if i<len(pool):result.append(pool[i])\n            if len(result)==count:break\n        if len(result)==before:raise ValueError(\'Insufficient pairs\')\n        i+=1\n    return result\n\ndef select_data(rows):\n    rng=random.Random(SEED);out={k:[] for k in [\'train\',\'development\',\'confirmation\']}\n    coverage={k:{} for k in out}\n    for domain in [\'real\',\'render\']:\n        train=groups([r for r in rows if r[\'split\']==\'train\' and r[\'subset\']==domain])\n        keys=list(train);rng.shuffle(keys)\n        picked=spread([train[k] for k in keys],256,rng);out[\'train\']+=picked\n        # Count membership in the FULL graph: sampling edges can disconnect a component.\n        membership={r[\'id\']:k for k,rr in train.items() for r in rr}\n        coverage[\'train\'][domain]=len({membership[r[\'id\']] for r in picked})\n        val=groups([r for r in rows if r[\'split\']==\'val\' and r[\'subset\']==domain])\n        keys=list(val);rng.shuffle(keys);cut=len(keys)//2\n        for part,kk in [(\'development\',keys[:cut]),(\'confirmation\',keys[cut:])]:\n            picked=spread([val[k] for k in kk],100,rng);out[part]+=picked\n            coverage[part][domain]=len(kk)\n    rng.shuffle(out[\'train\'])\n    for part in [\'development\',\'confirmation\']:out[part].sort(key=lambda r:r[\'id\'])\n    out.update(reward_rows=sorted([r for r in rows if r[\'split\']==\'train\'],key=lambda r:r[\'id\']),\n        coverage=coverage,config=dict(TRAIN_CONFIG),validation_previously_exposed=True)\n    out[\'selection_ids\']={k:identity(out[k]) for k in [\'train\',\'development\',\'confirmation\',\'reward_rows\']}\n    validate_bundle(out);return out\n\ndef validate_bundle(b):\n    if b[\'config\']!=TRAIN_CONFIG:raise ValueError(\'Training configuration changed\')\n    image_sets={}\n    for name,split,n in [(\'train\',\'train\',512),(\'development\',\'val\',200),(\'confirmation\',\'val\',200),(\'reward_rows\',\'train\',12452)]:\n        rows=b[name]\n        if len(rows)!=n or len({r[\'id\'] for r in rows})!=n or identity(rows)!=b[\'selection_ids\'][name]:raise ValueError(\'Selection identity/count mismatch: \'+name)\n        if name!=\'reward_rows\' and Counter(r[\'subset\'] for r in rows)!=Counter(real=n//2,render=n//2):raise ValueError(\'Unbalanced domain\')\n        for r in rows:\n            if r[\'split\']!=split or not r[\'id\'].startswith(r[\'subset\']+\'_\'+split+\'_\') or not r[\'caption\'].strip() or len(r[\'image\'])!=2:raise ValueError(\'Wrong split/reference\')\n            for im in r[\'image\']:\n                if safe_relative(im).parts[:2]!=(r[\'subset\'].capitalize(),split):raise ValueError(\'Image split mismatch\')\n        image_sets[name]={im for r in rows for im in r[\'image\']}\n    for a,c in [(\'train\',\'development\'),(\'train\',\'confirmation\'),(\'development\',\'confirmation\')]:\n        if image_sets[a]&image_sets[c]:raise ValueError(\'Image overlap: \'+a+\'/\'+c)\n    pool={r[\'id\']:r for r in b[\'reward_rows\']}\n    if any(pool.get(r[\'id\'])!=r for r in b[\'train\']):raise ValueError(\'Train pair absent from reward corpus\')\n\ndef expected_records(rows):return [dict(id=r[\'id\'],subset=r[\'subset\'],image=r[\'image\'],reference=r[\'caption\']) for r in rows]\ndef read_cache(folder,arm,meta,rows):\n    p=Path(folder)/(arm+\'.json\');ip=p.with_suffix(\'.identity.json\')\n    if p.exists() and not ip.exists():raise ValueError(\'Caption identity missing\')\n    if ip.exists() and read(ip)!=meta:raise ValueError(\'Wrong caption protocol/weights\')\n    cached=read(p) if p.exists() else []\n    if not isinstance(cached,list) or len(cached)>len(rows):raise ValueError(\'Caption count mismatch\')\n    for expected,record in zip(expected_records(rows),cached):\n        if any(record.get(k)!=v for k,v in expected.items()) or not isinstance(record.get(\'prediction\'),str):raise ValueError(\'Caption ID/image/reference mismatch\')\n    return cached\ndef write_cache(folder,arm,meta,rows,preds):\n    folder=Path(folder);require_identity(folder/(arm+\'.identity.json\'),meta)\n    if len(preds)>len(rows):raise ValueError(\'Too many predictions\')\n    records=[dict(e,**{k:p[k] for k in [\'prediction\',\'generation_tokens\',\'seconds\']}) for e,p in zip(expected_records(rows),preds)]\n    atomic(folder/(arm+\'.json\'),records)\n\ndef gate(metrics):\n    reasons=[]\n    for d in [\'All\',\'Real\',\'Render\']:\n        for a in ARMS:\n            for k in METRICS:\n                v=metrics.get(d,{}).get(a,{}).get(k)\n                if not isinstance(v,(int,float)) or not math.isfinite(v):reasons.append(\'Thiếu \'+d+\'/\'+a+\'/\'+k)\n    if reasons:return {\'passed\':False,\'reasons\':reasons,\'improvement_proven\':False}\n    for a in [\'G\',\'A\']:\n        if metrics[\'All\'][\'B\'][\'CIDEr\']-metrics[\'All\'][a][\'CIDEr\']<1.-1e-10:reasons.append(\'CIDEr All B chưa hơn \'+a+\' ít nhất1.0\')\n    for k in [\'BLEU-4\',\'METEOR\',\'ROUGE-L\']:\n        if metrics[\'All\'][\'B\'][k]<metrics[\'All\'][\'G\'][k]-1e-10:reasons.append(k+\' All giảm so G\')\n    for d in [\'Real\',\'Render\']:\n        if metrics[d][\'B\'][\'CIDEr\']<metrics[d][\'G\'][\'CIDEr\']-1e-10:reasons.append(\'CIDEr \'+d+\' giảm so G\')\n    return {\'passed\':not reasons,\'reasons\':reasons,\'improvement_proven\':False,\n        \'note\':\'Predeclared pilot gate; not a statistical significance test.\'}\n', 'scst_objective.py': '"""SCST log probability on exact sampled IDs after visual-token expansion."""\nimport torch\nimport torch.nn.functional as F\n\ndef sampled_layout(prompt,sample,eos,pad):\n    sample=list(map(int,sample))\n    if eos in sample:sample=sample[:sample.index(eos)+1]\n    if not sample or any(x<0 for x in sample):raise ValueError(\'Empty/invalid sampled tokens\')\n    if pad!=eos and pad in sample:raise ValueError(\'Padding before EOS in sampled sequence\')\n    if prompt.count(-200)!=2:raise ValueError(\'Need two image tokens\')\n    if len(prompt)+len(sample)+1458>4096:raise ValueError(\'No silent sequence truncation\')\n    return list(prompt)+sample,[-100]*len(prompt)+sample\n\ndef sequence_logprob(logits,expanded_labels):\n    if logits.shape[:2]!=expanded_labels.shape:raise ValueError(\'Labels must be image-expanded\')\n    targets=expanded_labels[:,1:];mask=targets!=-100\n    if not mask.any(dim=1).all():raise ValueError(\'No sampled answer tokens\')\n    # Select answer positions BEFORE vocabulary softmax to avoid allocating prompt×vocab gradients.\n    chosen=logits[:,:-1,:][mask].float();tokens=targets[mask]\n    values=-F.cross_entropy(chosen,tokens,reduction=\'none\')\n    batch_indices=torch.arange(logits.shape[0],device=logits.device)[:,None].expand_as(targets)[mask]\n    return torch.zeros(logits.shape[0],device=logits.device).scatter_add(0,batch_indices,values)\n\ndef scst_loss(logits,expanded_labels,advantage):\n    lp=sequence_logprob(logits,expanded_labels)\n    adv=torch.as_tensor(advantage,device=lp.device,dtype=lp.dtype).detach().reshape(-1)\n    if adv.shape!=lp.shape or not torch.isfinite(adv).all():raise ValueError(\'Invalid reward advantage\')\n    return -(adv*lp).mean()\n', 'scst_reward.py': '"""CIDEr-D with fixed document frequency from official TRAIN only; no minibatch IDF."""\nimport math\nfrom collections import defaultdict\nfrom pycocoevalcap.cider.cider_scorer import CiderScorer\n\nclass FixedFrequency(dict):\n    def __missing__(self,key):return 0.0 # Unknown generated n-grams must not mutate TRAIN IDF.\n\nclass FrozenCider:\n    def __init__(self,data):\n        self.document_count=data[\'document_count\']\n        if self.document_count<2:raise ValueError(\'Reward IDF needs a full training corpus\')\n        self.df=FixedFrequency({tuple(k):float(v) for k,v in data[\'document_frequency\']})\n        if not self.df or any(v<1 or v>self.document_count for v in self.df.values()):raise ValueError(\'Invalid document frequency\')\n    @classmethod\n    def from_tokenized(cls,refs):\n        s=CiderScorer(n=4,sigma=6.)\n        for values in refs.values():s+=(None,values)\n        s.compute_doc_freq()\n        return cls({\'document_count\':len(refs),\'document_frequency\':[[list(k),v] for k,v in sorted(s.document_frequency.items())]})\n    def to_json(self):return {\'document_count\':self.document_count,\'document_frequency\':[[list(k),v] for k,v in sorted(self.df.items())]}\n    def score_tokenized(self,pred,refs):\n        s=CiderScorer(test=pred,refs=refs,n=4,sigma=6.)\n        s.document_frequency=self.df;s.ref_len=math.log(self.document_count)\n        # compute_cider resets ref_len from len(crefs); override through a fixed-length wrapper.\n        class FixedCorpus(list):\n            def __len__(self):return self.document_count\n        wrapped=FixedCorpus(s.crefs);wrapped.document_count=self.document_count;s.crefs=wrapped\n        value=float(s.compute_cider()[0])\n        if not math.isfinite(value):raise ValueError(\'Nonfinite CIDEr reward\')\n        return value\n', 'scst_state.py': '"""Committed two-slot checkpoints and crash-safe cumulative budget."""\nimport contextlib,os,random,time,signal,subprocess,sys,json,queue,threading\nfrom pathlib import Path\nfrom scst_support import atomic,read,sha\n\nclass Checkpoints:\n    def __init__(self,folder,arm,contract):\n        self.folder=Path(folder);self.arm=arm;self.contract=contract\n        self.pointer=self.folder/\'commit.json\'\n    def metadata(self):\n        if not self.pointer.exists():return None\n        p=read(self.pointer)\n        if p[\'arm\']!=self.arm or p[\'contract\']!=self.contract or p[\'slot\'] not in [0,1]:raise ValueError(\'Wrong checkpoint identity\')\n        if sha(self.folder/f"slot{p[\'slot\']}.pt")!=p[\'sha256\']:raise ValueError(\'Committed checkpoint is corrupted; do not restart silently\')\n        return p\n    def load(self):\n        import torch\n        p=self.metadata()\n        if p is None:return None\n        # Only this notebook\'s SHA-verified self-generated checkpoint is deserialized.\n        data=torch.load(self.folder/f"slot{p[\'slot\']}.pt",map_location=\'cpu\',weights_only=False)\n        if data[\'arm\']!=self.arm or data[\'contract\']!=self.contract or data[\'step\']!=p[\'step\']:raise ValueError(\'Checkpoint payload identity mismatch\')\n        return data\n    def save(self,data):\n        import torch\n        old=self.metadata()\n        if old and data[\'step\']<old[\'step\']:raise ValueError(\'Cannot replace checkpoint with an older step\')\n        slot=1-old[\'slot\'] if old else 0;self.folder.mkdir(parents=True,exist_ok=True)\n        target=self.folder/f\'slot{slot}.pt\';temp=target.with_suffix(\'.tmp\')\n        payload=dict(data,arm=self.arm,contract=self.contract)\n        torch.save(payload,temp);digest=sha(temp);os.replace(temp,target)\n        atomic(self.pointer,dict(arm=self.arm,contract=self.contract,slot=slot,step=data[\'step\'],sha256=digest))\n\ndef rng_state():\n    import torch\n    return {\'python\':random.getstate(),\'cpu\':torch.get_rng_state(),\'cuda\':torch.cuda.get_rng_state_all() if torch.cuda.is_available() else []}\n\ndef restore_rng(s):\n    import torch\n    random.setstate(s[\'python\']);torch.set_rng_state(s[\'cpu\'])\n    if s[\'cuda\']:torch.cuda.set_rng_state_all(s[\'cuda\'])\n\nclass LeaseWatchdog:\n    """Separate process enforces a prepaid block even while CUDA/Java blocks Python."""\n    def __init__(self):\n        if os.name!=\'nt\' and os.getpgrp()!=os.getpid():raise RuntimeError(\'Run the GPU worker in its own process session before enforcing budget\')\n        self.proc=subprocess.Popen([sys.executable,str(Path(__file__).resolve()),\'--watchdog\',str(os.getpid())],\n            stdin=subprocess.PIPE,stdout=subprocess.PIPE,text=True,bufsize=1,start_new_session=(os.name!=\'nt\'))\n    def send(self,deadline):\n        self.proc.stdin.write(json.dumps(deadline)+\'\\n\');self.proc.stdin.flush()\n        if self.proc.stdout.readline().strip()!=\'ok\':raise RuntimeError(\'Budget watchdog unavailable; stop instead of running unmetered\')\n    def arm(self,seconds):self.send(time.monotonic()+max(.001,seconds))\n    def disarm(self):self.send(None)\n\ndef watchdog(pid):\n    messages=queue.Queue()\n    def reader():\n        for line in sys.stdin:messages.put(json.loads(line))\n        messages.put(\'closed\')\n    threading.Thread(target=reader,daemon=True).start();deadline=None\n    while True:\n        try:message=messages.get(timeout=None if deadline is None else max(0.,deadline-time.monotonic()))\n        except queue.Empty:\n            try:\n                if os.name==\'nt\':os.kill(pid,signal.SIGTERM)\n                else:os.killpg(os.getpgid(pid),signal.SIGKILL)\n            except ProcessLookupError:pass\n            return\n        if message==\'closed\':return\n        # Do not accept a late cancellation/refund after the prepaid lease expired.\n        if deadline is not None and time.monotonic()>=deadline:\n            try:\n                if os.name==\'nt\':os.kill(pid,signal.SIGTERM)\n                else:os.killpg(os.getpgid(pid),signal.SIGKILL)\n            except ProcessLookupError:pass\n            return\n        deadline=message;print(\'ok\',flush=True)\n\n@contextlib.contextmanager\ndef scoring_signals():\n    def interrupt(signum,frame):raise InterruptedError(\'Scoring interrupted; cleaning METEOR before exit\')\n    old={s:signal.getsignal(s) for s in [signal.SIGTERM,signal.SIGINT]}\n    for s in old:signal.signal(s,interrupt)\n    try:yield\n    finally:\n        for s,handler in old.items():signal.signal(s,handler)\n\nclass Budget:\n    def __init__(self,path,limit,clock=time.monotonic,enforce=False):\n        self.path=Path(path);self.clock=clock;self.started=None;self.reserved=0;self.last=clock()\n        self.data=read(path) if self.path.exists() else {\'limit\':limit,\'spent\':0.,\'active\':None,\'blocks\':[]}\n        if self.data[\'limit\']!=limit:raise ValueError(\'Cannot extend the locked budget\')\n        # A lost process already reserved its block. Never refund unobserved work.\n        if self.data[\'active\']:\n            self.data[\'blocks\'].append(dict(self.data[\'active\'],crash_charged=True))\n            self.data[\'active\']=None;atomic(self.path,self.data)\n        self.guard=LeaseWatchdog() if enforce else None\n    def remaining(self):\n        if self.started is None:\n            now=self.clock();gap=max(0.,now-self.last);self.last=now\n            if gap:\n                self.data[\'spent\']+=gap;atomic(self.path,self.data)\n        return max(0.,self.data[\'limit\']-self.data[\'spent\'])\n    def begin(self,reserve,label=\'work\'):\n        if self.started is not None:raise RuntimeError(\'Nested budget block\')\n        if reserve<=0 or reserve>self.remaining():raise TimeoutError(\'Đã chạm ngân sách đã khóa. Chạy ô 4 tải báo cáo; không tự chạy lại từ đầu.\')\n        self.reserved=reserve;self.started=self.clock();self.data[\'spent\']+=reserve\n        self.data[\'active\']={\'label\':label,\'reserved_seconds\':reserve};atomic(self.path,self.data)\n        if self.guard:self.guard.arm(reserve-(self.clock()-self.started))\n    def close(self):\n        if self.started is None:return\n        if self.guard:self.guard.disarm()\n        used=max(0.,self.clock()-self.started);entry=self.data[\'active\']\n        self.data[\'spent\']+=used-self.reserved\n        self.data[\'blocks\'].append(dict(entry,actual_seconds=used));self.data[\'active\']=None\n        self.started=None;self.reserved=0;self.last=self.clock();atomic(self.path,self.data)\n    @contextlib.contextmanager\n    def block(self,reserve,label):\n        self.begin(reserve,label)\n        try:yield\n        finally:self.close()\n\nif __name__==\'__main__\' and len(sys.argv)==3 and sys.argv[1]==\'--watchdog\':watchdog(int(sys.argv[2]))\n', 'scst_prepare.py': '"""Stage original weights, selected train/val images and TRAIN-only fixed reward."""\nimport concurrent.futures\nimport json\nimport sys\nfrom pathlib import Path\nfrom scst_support import *\n\ndef prepare(cfg):\n    from huggingface_hub import HfApi,hf_hub_download,snapshot_download\n    from PIL import Image\n    work=Path(cfg[\'work\']);out=Path(cfg[\'out\']);storage=Path(cfg[\'storage\'])\n    bundle=read(work/\'bundle.json\');validate_bundle(bundle)\n    rows=bundle[\'train\']+bundle[\'development\']+bundle[\'confirmation\']\n    require_identity(out/\'selection.json\',{k:bundle[k] for k in [\'train\',\'development\',\'confirmation\',\'coverage\',\'selection_ids\']})\n    hashes=bundle[\'prior_protocol\'][\'checkpoint\'];drive_model=storage/\'M3Diff_checkpoint\';model=Path(cfg[\'model\'])\n    for i,(name,digest) in enumerate(hashes.items(),1):\n        safe_relative(name);src=drive_model/name\n        if not src.is_file():raise FileNotFoundError(\'Thiếu trọng số gốc trên Drive: \'+str(src))\n        print(f\'Checkpoint gốc: {i}/{len(hashes)} — {name}\',flush=True)\n        copy_checked(src,model/name,digest)\n    vision=bundle[\'prior_protocol\'][\'vision\']\n    print(\'Chuẩn bị SigLIP đúng revision đã chạy.\',flush=True)\n    vision_path=snapshot_download(vision[\'repo\'],revision=vision[\'revision\'],\n        allow_patterns=[\'*.json\',\'*.safetensors\',\'pytorch_model.bin\'],cache_dir=str(storage/\'hf_cache\'))\n    # Stage vision locally too: inference reads local disk, not Drive.\n    local_vision=work/\'vision\';local_vision.mkdir(exist_ok=True);vision_hashes={}\n    for src in sorted(Path(vision_path).iterdir()):\n        if src.is_file():\n            digest=sha(src);copy_checked(src,local_vision/src.name,digest);vision_hashes[src.name]=digest\n    if vision_hashes!=bundle[\'vision_hashes\']:raise ValueError(\'SigLIP checkpoint differs from verified original\')\n    need=sorted({name for r in rows for name in r[\'image\']})\n    digest_file=out/\'hub_image_digests.json\'\n    if digest_file.exists():\n        record=read(digest_file)\n        if record[\'revision\']!=REVISION:raise ValueError(\'Wrong image revision\')\n        digests=record[\'digests\']\n    else:\n        digests={}\n        print(\'Lấy checksum ảnh từ đúng revision OmniDiff.\',flush=True)\n        api=HfApi()\n        for subtree in [\'Real/train/images\',\'Render/train/images\',\'Real/val/images\',\'Render/val/images\']:\n            for item in api.list_repo_tree(\'IVC-liuyuan/OmniDiff\',repo_type=\'dataset\',revision=REVISION,path_in_repo=subtree,recursive=True):\n                if item.path not in need:continue\n                lfs=getattr(item,\'lfs\',None)\n                value=(lfs.get(\'sha256\') if isinstance(lfs,dict) else getattr(lfs,\'sha256\',None)) if lfs else None\n                digests[item.path]={\'kind\':\'sha256\' if value else \'git-sha1\',\'value\':value or item.blob_id}\n        if set(digests)!=set(need):raise ValueError(\'Missing image digests at pinned revision\')\n        atomic(digest_file,{\'revision\':REVISION,\'digests\':digests})\n    if set(digests)!=set(need):raise ValueError(\'Image digest manifest incomplete\')\n    images=Path(cfg[\'images\']);drive_images=storage/\'SCST_LoRA_v1/images\'\n    def one(name):\n        safe_relative(name);target=drive_images/name;digest=digests[name];reused=False\n        candidates=[target,storage/\'MDP_Projector_v1/images\'/name,storage/\'M3Diff_Data\'/name,storage/\'M3Diff_Data/OmniDiff\'/name]\n        for candidate in candidates:\n            if candidate.is_file() and verify_hub_digest(candidate,digest):\n                if candidate!=target:copy_checked(candidate,target,sha(candidate))\n                reused=True;break\n        if not reused:\n            src=Path(hf_hub_download(\'IVC-liuyuan/OmniDiff\',name,repo_type=\'dataset\',revision=REVISION))\n            if not verify_hub_digest(src,digest):raise ValueError(\'Wrong Hub image: \'+name)\n            copy_checked(src,target,sha(src))\n        if not verify_hub_digest(target,digest):raise ValueError(\'Wrong Drive image: \'+name)\n        with Image.open(target) as im:size=list(im.size);im.verify()\n        checksum=sha(target);copy_checked(target,images/name,checksum)\n        return {\'path\':name,\'sha256\':checksum,\'bytes\':target.stat().st_size,\'size\':size},reused\n    manifest=[];reused_count=0\n    with concurrent.futures.ThreadPoolExecutor(max_workers=4) as pool:\n        for item,reused in pool.map(one,need):\n            manifest.append(item);reused_count+=int(reused)\n            if len(manifest)%25==0 or len(manifest)==len(need):\n                print(f\'Ảnh: {len(manifest)}/{len(need)}; dùng lại {reused_count}.\',flush=True)\n    require_identity(out/\'image_manifest.json\',manifest)\n    require_identity(out/\'vision_manifest.json\',vision_hashes)\n    from caption_metrics import tokenize_checked\n    from scst_reward import FrozenCider\n    reward_path=out/\'reward.json\'\n    if reward_path.exists():\n        rd=read(reward_path)\n        if rd[\'selection\']!=bundle[\'selection_ids\'][\'reward_rows\'] or rd[\'scorer\'][\'document_count\']!=12452:raise ValueError(\'Wrong stored reward corpus\')\n        if rd[\'train_selection\']!=bundle[\'selection_ids\'][\'train\']:raise ValueError(\'Wrong train references\')\n    else:\n        print(\'Tính IDF cho CIDEr từ 12.452 cặp TRAIN; không dùng validation/test.\',flush=True)\n        tokenized=tokenize_checked({r[\'id\']:[{\'caption\':r[\'caption\']}] for r in bundle[\'reward_rows\']})\n        scorer=FrozenCider.from_tokenized(tokenized)\n        atomic(reward_path,{\'selection\':bundle[\'selection_ids\'][\'reward_rows\'],\'train_selection\':bundle[\'selection_ids\'][\'train\'],\n            \'scorer\':scorer.to_json(),\'train_tokenized\':{r[\'id\']:tokenized[r[\'id\']] for r in bundle[\'train\']}})\n    require_identity(out/\'reward_identity.json\',{\'sha256\':sha(reward_path),\'corpus\':bundle[\'selection_ids\'][\'reward_rows\']})\n    atomic(work/\'prepared.json\',dict(cfg,vision=str(local_vision),vision_hashes=vision_hashes,\n        image_manifest=manifest,checkpoint_identity=hashes))\n    atomic(out/\'data_ready.json\',{\'ready\':True,\'train_pairs\':512,\'development_pairs\':200,\'confirmation_pairs\':200,\n        \'unique_images\':len(need),\'reused_images\':reused_count,\'training_ran\':False,\'reward_sha256\':sha(reward_path)})\n    print(\'Ô 2 hoàn tất. Chạy ô 3: smoke → G/A/B → xác nhận nếu đạt.\',flush=True)\n\nif __name__==\'__main__\':prepare(read(sys.argv[1]))\n', 'scst_runner.py': '"""One GPU model, independent CE/CE+SCST adapters, locked pilot and resumable state."""\nimport contextlib,importlib.metadata,math,random,signal,sys,time,traceback\nfrom pathlib import Path\nfrom scst_support import *\nfrom scst_state import Budget,Checkpoints,rng_state,restore_rng,scoring_signals\nfrom inference_tokens import token_layout\nfrom scst_objective import sampled_layout,scst_loss\n\ndef run(cfg):\n    import torch\n    from PIL import Image\n    from peft import LoraConfig,get_peft_model,get_peft_model_state_dict,set_peft_model_state_dict\n    from transformers import AutoTokenizer,GenerationConfig\n    from llava.model.language_model.llava_qwen import LlavaQwenForCausalLM,LlavaQwenConfig\n    from llava.mm_utils import process_images\n    from caption_metrics import tokenize_checked\n    from scst_reward import FrozenCider\n    from scst_score import score_part,verify_run,cache_meta,completed_run\n    out=Path(cfg[\'out\']);work=Path(cfg[\'work\']);images=Path(cfg[\'images\'])\n    b=read(work/\'bundle.json\');validate_bundle(b);tc=b[\'config\'];prior=b[\'prior_protocol\']\n    if completed_run(out,b,work):\n        print(\'Lượt này đã kết thúc đúng cấu hình. Không nạp model; chạy ô 4.\',flush=True);return\n    if should_stop_before_model(read(out/\'eta.json\') if (out/\'eta.json\').exists() else None,\n        read(out/\'budget.json\') if (out/\'budget.json\').exists() else None):\n        raise TimeoutError(\'Lượt trước đã dừng vì ngân sách/dự toán. Không nạp model lại; chạy ô 4 gửi ZIP.\')\n    if not torch.cuda.is_available() or not torch.cuda.is_bf16_supported() or torch.cuda.device_count()!=1:raise RuntimeError(\'Cần một GPU A100 80GB hỗ trợ BF16.\')\n    gpu=torch.cuda.get_device_properties(0)\n    if \'A100\' not in gpu.name or gpu.total_memory<75*1024**3:raise RuntimeError(\'Pilot này khóa A100 80GB; chưa đo trên GPU khác.\')\n    for name,digest in prior[\'checkpoint\'].items():\n        if sha(Path(cfg[\'model\'])/name)!=digest:raise ValueError(\'Original checkpoint checksum mismatch\')\n    if cfg[\'vision_hashes\']!=b[\'vision_hashes\']:raise ValueError(\'Vision differs from the verified baseline\')\n    for name,digest in cfg[\'vision_hashes\'].items():\n        if sha(Path(cfg[\'vision\'])/name)!=digest:raise ValueError(\'Local vision checksum mismatch\')\n    for item in cfg[\'image_manifest\']:\n        if sha(images/item[\'path\'])!=item[\'sha256\']:raise ValueError(\'Local image checksum mismatch\')\n    actual={n:importlib.metadata.version(n) for n in b[\'packages\']}\n    if actual!=b[\'packages\']:raise ValueError(\'Wrong package versions: \'+str(actual))\n    protocol={\'config\':tc,\'selection\':b[\'selection_ids\'],\'checkpoint\':prior[\'checkpoint\'],\n        \'vision\':prior[\'vision\'],\'vision_hashes\':cfg[\'vision_hashes\'],\'annotation_sha\':b[\'annotation_sha\'],\n        \'image_manifest\':identity(cfg[\'image_manifest\']),\'source_sha\':cfg[\'source_sha\'],\n        \'code\':{p.name:sha(p) for p in sorted(work.glob(\'*.py\'))},\'packages\':actual,\n        \'precision\':prior[\'precision\'],\'decode\':prior[\'decode\'],\'question\':QUESTION,\n        \'gpu\':gpu.name,\'cuda\':torch.version.cuda,\'cudnn\':torch.backends.cudnn.version(),\n        \'reward_sha\':sha(out/\'reward.json\'),\'reward_data\':b[\'selection_ids\'][\'reward_rows\'],\n        \'reward_description\':\'CIDEr-D raw 0..10, fixed DF over all 12452 official train pairs; PTB\',\n        \'sampling\':{\'temperature\':1.,\'top_p\':1.,\'top_k\':0,\'num_beams\':1,\'max_new_tokens\':256},\n        \'validation_previously_exposed\':True,\'test_used\':False,\'training_goal\':\'pilot, no promised gain\'}\n    require_identity(out/\'protocol.json\',protocol);contract=identity(protocol)\n    stores={a:Checkpoints(out/\'checkpoints\'/a,a,contract) for a in [\'A\',\'B\']}\n    for store in stores.values():store.metadata() # fail before loading if an old commit is bad\n    budget=Budget(out/\'budget.json\',tc[\'budget_seconds\'],enforce=True);deadline=time.monotonic()+budget.remaining()\n    stopping=[False]\n    def request_stop(signum,frame):stopping[0]=True\n    signal.signal(signal.SIGTERM,request_stop);signal.signal(signal.SIGINT,request_stop)\n    def can_work(reserve):\n        if stopping[0]:raise InterruptedError(\'Đã dừng mềm và giữ checkpoint; chạy ô 4 hoặc tiếp tục ô 3.\')\n        if time.monotonic()+reserve>deadline or reserve>budget.remaining():raise TimeoutError(\'Đã hết phần ngân sách còn lại; chạy ô 4 gửi báo cáo.\')\n    config=LlavaQwenConfig.from_pretrained(cfg[\'model\'],local_files_only=True)\n    if config.mm_resampler_type!=\'mdp\' or config.image_aspect_ratio!=\'pad\' or getattr(config,\'attention_dropout\',0.)!=0.:raise ValueError(\'Wrong architecture/dropout\')\n    config.unfreeze_mm_vision_tower=True;config.mm_vision_tower=cfg[\'vision\'];config.use_cache=False;config.tokenizer_model_max_length=4096\n    def seed():random.seed(SEED);torch.manual_seed(SEED);torch.cuda.manual_seed_all(SEED)\n    seed();torch.backends.cuda.matmul.allow_tf32=False\n    can_work(600)\n    with budget.block(600,\'load_model\'):\n        print(\'Nạp một model vào GPU. G/A/B dùng chung model này và đổi adapter.\',flush=True)\n        base,info=LlavaQwenForCausalLM.from_pretrained(cfg[\'model\'],config=config,local_files_only=True,\n            torch_dtype=torch.bfloat16,device_map={\'\':0},attn_implementation=\'sdpa\',low_cpu_mem_usage=True,output_loading_info=True)\n        atomic(out/\'loading_info.json\',info)\n        if any(info.get(k) for k in [\'missing_keys\',\'unexpected_keys\',\'mismatched_keys\',\'error_msgs\']):raise ValueError(\'Checkpoint loading mismatch\')\n        vision=base.get_vision_tower()\n        if not vision.is_loaded:raise ValueError(\'Vision not loaded\')\n        vision.to(device=\'cuda\',dtype=torch.bfloat16);inner=base.get_model()\n        inner.vision_resampler.float();inner.mm_projector.float();base.requires_grad_(False)\n        targets=[n for n,m in base.named_modules() if n.startswith(\'model.layers.\') and n.endswith((\'.q_proj\',\'.v_proj\')) and isinstance(m,torch.nn.Linear)]\n        if len(targets)!=2*config.num_hidden_layers:raise ValueError(\'Wrong LoRA targets\')\n        model=get_peft_model(base,LoraConfig(r=16,lora_alpha=32,lora_dropout=0.,target_modules=targets,bias=\'none\',task_type=\'CAUSAL_LM\'))\n        trainable={n:p for n,p in model.named_parameters() if p.requires_grad}\n        if not trainable or any(\'lora_\' not in n for n in trainable):raise ValueError(\'Only LoRA may train\')\n        for p in trainable.values():p.data=p.data.float()\n        model.enable_input_require_grads();model.gradient_checkpointing_enable()\n        tok=AutoTokenizer.from_pretrained(cfg[\'model\'],use_fast=False,local_files_only=True)\n        end=tok.convert_tokens_to_ids(\'<|im_end|>\');pad=end # no separate PAD with probability in the sampled policy\n        def adapter():return {k:v.detach().cpu().clone() for k,v in get_peft_model_state_dict(model).items()}\n        initial=adapter()\n        def restore(state):\n            if set(state)!=set(initial) or any(state[k].shape!=initial[k].shape for k in initial):raise ValueError(\'Wrong adapter keys/shapes\')\n            set_peft_model_state_dict(model,state)\n            if any(not torch.equal(v,state[k].cpu()) for k,v in adapter().items()):raise ValueError(\'Adapter reload mismatch\')\n        def fingerprint(module):\n            h=hashlib.sha256()\n            for k,v in module.state_dict().items():h.update(k.encode());h.update(v.detach().cpu().contiguous().view(torch.uint8).numpy().tobytes())\n            return h.hexdigest()\n        frozen={\'mdp\':fingerprint(inner.vision_resampler),\'projector\':fingerprint(inner.mm_projector)}\n        require_identity(out/\'trainable.json\',{\'targets\':targets,\'parameters\':sum(p.numel() for p in trainable.values()),\'initial\':fingerprint(model.base_model.model.model.layers[0].self_attn.q_proj.lora_A),\'frozen\':frozen})\n        require_identity(out/\'weights_G.json\',{\'protocol\':contract,\'weights\':identity(prior[\'checkpoint\'])})\n    reward_data=read(out/\'reward.json\')\n    if reward_data[\'selection\']!=b[\'selection_ids\'][\'reward_rows\']:raise ValueError(\'Wrong reward corpus\')\n    reward=FrozenCider(reward_data[\'scorer\']);refs=reward_data[\'train_tokenized\']\n    if reward.document_count!=12452 or set(refs)!={r[\'id\'] for r in b[\'train\']}:raise ValueError(\'Wrong reward references\')\n    def conv(row):return {\'conversations\':[{\'from\':\'human\',\'value\':QUESTION},{\'from\':\'gpt\',\'value\':row[\'caption\']}]}\n    def image_input(row):\n        parts=[]\n        for name in row[\'image\']:\n            with Image.open(images/name) as im:parts.append(process_images([im.convert(\'RGB\')],vision.image_processor,config).to(device=\'cuda\',dtype=torch.bfloat16))\n        return [torch.cat(parts,dim=0)]\n    def generate(row,pixels,sample=False):\n        model.eval();prompt,_=token_layout(tok,conv(row),False)\n        # A clean GenerationConfig avoids inherited top-k, penalties or forced-token processors.\n        gc=GenerationConfig(do_sample=sample,num_beams=1,max_new_tokens=256,temperature=1.,top_p=1.,top_k=0,\n            repetition_penalty=1.,no_repeat_ngram_size=0,min_new_tokens=0,pad_token_id=pad,eos_token_id=end,\n            bos_token_id=tok.bos_token_id,use_cache=True,return_dict_in_generate=False,output_scores=False)\n        with torch.inference_mode(),torch.autocast(\'cuda\',dtype=torch.bfloat16):\n            ids=model.generate(inputs=torch.tensor([prompt],device=\'cuda\'),attention_mask=torch.ones((1,len(prompt)),dtype=torch.long,device=\'cuda\'),\n                images=pixels,generation_config=gc)\n        ids=ids[0].detach().cpu().tolist()\n        if not ids or len(ids)>256:raise ValueError(\'Unexpected generated token layout\')\n        return ids,tok.decode(ids,skip_special_tokens=True).strip(),prompt\n    def forward(ids,labels,pixels):\n        model.train();vision.eval();inner.vision_resampler.eval();inner.mm_projector.eval()\n        x=torch.tensor([ids],device=\'cuda\');y=torch.tensor([labels],device=\'cuda\');mask=torch.ones_like(x)\n        with torch.no_grad(),torch.autocast(\'cuda\',dtype=torch.bfloat16):\n            _,pos,attn,_,emb,expanded=base.prepare_inputs_labels_for_multimodal(x,None,mask,None,y,pixels)\n        if int((expanded!=-100).sum())!=sum(i!=-100 for i in labels):raise ValueError(\'Answer labels lost during image expansion\')\n        emb=emb.detach().requires_grad_(True)\n        with torch.autocast(\'cuda\',dtype=torch.bfloat16):\n            result=model(inputs_embeds=emb,attention_mask=attn,position_ids=pos,labels=expanded,use_cache=False,return_dict=True)\n        return result,expanded\n    def update(arm,rows,optim):\n        start=time.monotonic();optim.zero_grad(set_to_none=True);logs=[]\n        for row in rows:\n            pixels=image_input(row);ids,labels=token_layout(tok,conv(row),True)\n            result,_=forward(ids,labels,pixels);ce=result.loss\n            if not torch.isfinite(ce):raise ValueError(\'Nonfinite CE\')\n            (ce/len(rows)).backward();entry={\'id\':row[\'id\'],\'ce\':float(ce.detach())};del result,ce\n            if arm==\'B\':\n                sampled,text,prompt=generate(row,pixels,True);_,greedy,_=generate(row,pixels,False)\n                tokens=tokenize_checked({\'sample\':[{\'caption\':text}],\'greedy\':[{\'caption\':greedy}]})\n                rs=reward.score_tokenized(tokens[\'sample\'][0],refs[row[\'id\']]);rg=reward.score_tokenized(tokens[\'greedy\'][0],refs[row[\'id\']]);adv=rs-rg\n                x,y=sampled_layout(prompt,sampled,end,pad);result,expanded=forward(x,y,pixels)\n                rl=scst_loss(result.logits,expanded,torch.tensor([adv],device=\'cuda\'))\n                if not torch.isfinite(rl):raise ValueError(\'Nonfinite SCST loss\')\n                (tc[\'scst_weight\']*rl/len(rows)).backward()\n                entry.update(sample_reward=rs,greedy_reward=rg,advantage=adv,rl=float(rl.detach()),tokens=len(sampled),sample=text,greedy=greedy)\n                del result,expanded,rl\n            logs.append(entry);del pixels\n        if any(p.grad is not None for n,p in model.named_parameters() if n not in trainable):raise ValueError(\'Frozen weight received gradient\')\n        grads=[p.grad for p in trainable.values() if p.grad is not None]\n        if not grads or any(not torch.isfinite(g).all() for g in grads) or not any(g.abs().max()>0 for g in grads):raise ValueError(\'Invalid LoRA gradients\')\n        norm=torch.nn.utils.clip_grad_norm_(list(trainable.values()),tc[\'clip_grad\']);optim.step();torch.cuda.synchronize()\n        return {\'seconds\':time.monotonic()-start,\'grad_norm\':float(norm),\'rows\':logs}\n    def optimizer():return torch.optim.AdamW(list(trainable.values()),lr=tc[\'lr\'],weight_decay=0.)\n    def check_frozen():\n        if frozen!={\'mdp\':fingerprint(inner.vision_resampler),\'projector\':fingerprint(inner.mm_projector)}:raise ValueError(\'Frozen modules changed\')\n    smoke_path=out/\'smoke.json\'\n    if not smoke_path.exists():\n        smoke={\'protocol\':contract,\'training_discarded\':True,\'updates\':{},\'generation_seconds\':[]}\n        for arm in [\'A\',\'B\']:\n            restore(initial);seed();optim=optimizer();steps=[]\n            for i in range(2):\n                can_work(300)\n                with budget.block(300,\'smoke_\'+arm):steps.append(update(arm,b[\'train\'][i*4:i*4+4],optim))\n            if all(torch.equal(v,initial[k]) for k,v in adapter().items()):raise ValueError(\'Smoke did not change LoRA\')\n            temporary=Checkpoints(out/\'checkpoints\'/(\'smoke_\'+arm),arm,contract)\n            with budget.block(60,\'smoke_save_reload\'):\n                temporary.save({\'step\':2,\'adapter\':adapter(),\'optimizer\':optim.state_dict(),\'rng\':rng_state(),\'order\':[r[\'id\'] for r in b[\'train\']]})\n                saved=temporary.load();restore(initial);restore(saved[\'adapter\']);optim.load_state_dict(saved[\'optimizer\']);restore_rng(saved[\'rng\'])\n            smoke[\'updates\'][arm]=steps;del optim\n        if not any(abs(r[\'advantage\'])>1e-8 for s in smoke[\'updates\'][\'B\'] for r in s[\'rows\']):raise ValueError(\'Smoke SCST advantage is zero for every pair; stop for diagnosis\')\n        restore(initial);seed();check_frozen()\n        for row in b[\'train\'][:4]:\n            can_work(60)\n            with budget.block(60,\'smoke_generate\'):\n                start=time.monotonic();generate(row,image_input(row));torch.cuda.synchronize();smoke[\'generation_seconds\'].append(time.monotonic()-start)\n        smoke[\'peak_gb\']=torch.cuda.max_memory_allocated()/1024**3;atomic(smoke_path,smoke)\n    smoke=read(smoke_path)\n    if smoke[\'protocol\']!=contract:raise ValueError(\'Wrong smoke identity\')\n    restore(initial);seed();check_frozen()\n    rate={a:sum(s[\'seconds\'] for s in smoke[\'updates\'][a])/len(smoke[\'updates\'][a]) for a in [\'A\',\'B\']}\n    gen_rate=sum(smoke[\'generation_seconds\'])/len(smoke[\'generation_seconds\'])\n    if not (out/\'pilot_admitted.json\').exists():\n        # Budget worst case includes confirmation, both arms and scoring; intentionally conservative.\n        estimate=1.4*(128*(rate[\'A\']+rate[\'B\'])+1200*gen_rate)+600\n        admission={\'protocol\':contract,\'estimated_seconds\':estimate,\'remaining_seconds\':budget.remaining(),\n            \'seconds_per_update\':rate,\'seconds_per_caption\':gen_rate,\'admitted\':estimate<=budget.remaining(),\n            \'note\':\'Measurement from a small smoke; estimate is uncertain, not guaranteed.\'}\n        atomic(out/\'eta.json\',admission)\n        print(f"Đo thực tế: dự kiến phần còn lại khoảng {estimate/60:.0f} phút; ngân sách còn {budget.remaining()/60:.0f} phút.",flush=True)\n        if not admission[\'admitted\']:raise TimeoutError(\'Smoke đạt nhưng dự toán vượt ngân sách. Dừng trước pilot; chạy ô 4 gửi ZIP để quyết định, không tự tăng thời gian.\')\n        atomic(out/\'pilot_admitted.json\',{\'protocol\':contract})\n    elif read(out/\'pilot_admitted.json\')!={\'protocol\':contract}:raise ValueError(\'Pilot admission identity mismatch\')\n    def train_arm(arm):\n        store=stores[arm];state=store.load();restore(initial);seed();optim=optimizer();step=0;history=[]\n        if state:\n            if state[\'order\']!=[r[\'id\'] for r in b[\'train\']] or not 0<=state[\'step\']<=128:raise ValueError(\'Wrong resumed order/step\')\n            restore(state[\'adapter\']);optim.load_state_dict(state[\'optimizer\']);restore_rng(state[\'rng\']);step=state[\'step\'];history=state[\'history\']\n        def save():store.save({\'step\':step,\'adapter\':adapter(),\'optimizer\':optim.state_dict(),\'rng\':rng_state(),\n                              \'order\':[r[\'id\'] for r in b[\'train\']],\'history\':history})\n        while step<128:\n            reserve=max(30,rate[arm]*2+15)\n            try:can_work(reserve)\n            except (TimeoutError,InterruptedError):\n                with budget.block(min(30,budget.remaining()),\'stop_save_\'+arm):save()\n                raise\n            with budget.block(reserve,\'update_\'+arm):\n                log=update(arm,b[\'train\'][step*4:step*4+4],optim);step+=1;history.append(dict(step=step,**log))\n                if step%5==0 or step==128 or stopping[0]:save()\n            print(f\'{arm}: {step}/128 cập nhật; {log["seconds"]:.1f}s; còn {budget.remaining()/60:.1f} phút.\',flush=True)\n        check_frozen();meta=store.metadata()\n        require_identity(out/f\'weights_{arm}.json\',{\'protocol\':contract,\'weights\':meta[\'sha256\'],\'step\':128})\n        atomic(out/f\'train_{arm}.json\',{\'protocol\':contract,\'updates\':128,\'pairs\':512,\'history\':history,\n            \'seconds\':sum(s[\'seconds\'] for s in history),\'peak_gb\':torch.cuda.max_memory_allocated()/1024**3})\n        del optim,state\n    def evaluate(arm,part):\n        if arm!=\'G\':\n            state=stores[arm].load()\n            if state is None or state[\'step\']!=128:raise ValueError(\'Evaluation needs the FINAL adapter\')\n            restore(state[\'adapter\']);del state\n        else:restore(initial)\n        rows=b[part];meta=cache_meta(out,b,protocol,part,arm);folder=out/part\n        cache=read_cache(folder,arm,meta,rows)\n        with model.disable_adapter() if arm==\'G\' else contextlib.nullcontext():\n            for row in rows[len(cache):]:\n                reserve=max(30,3*gen_rate+5);can_work(reserve)\n                with budget.block(reserve,\'caption_\'+part+\'_\'+arm):\n                    start=time.monotonic();ids,pred,_=generate(row,image_input(row));torch.cuda.synchronize()\n                    cache.append({\'prediction\':pred,\'generation_tokens\':len(ids),\'seconds\':time.monotonic()-start})\n                    write_cache(folder,arm,meta,rows,cache)\n                if len(cache)%20==0:print(f\'{part} — {arm}: {len(cache)}/{len(rows)}\',flush=True)\n    evaluate(\'G\',\'development\')\n    for arm in [\'A\',\'B\']:train_arm(arm);evaluate(arm,\'development\')\n    can_work(700)\n    with budget.block(700,\'score_development\'),scoring_signals():result=score_part(out,b,\'development\')\n    if result[\'status\']!=\'complete\':raise RuntimeError(\'Thiếu metric; chạy ô 4 chấm lại, không mở confirmation khi thiếu điểm.\')\n    decision=gate(result[\'metrics\']);atomic(out/\'development_gate.json\',dict(decision,protocol=contract))\n    if decision[\'passed\']:\n        print(\'Development đạt ngưỡng đã khóa. Đánh giá thêm 200 cặp confirmation, không học thêm.\',flush=True)\n        for arm in ARMS:evaluate(arm,\'confirmation\')\n        can_work(700)\n        with budget.block(700,\'score_confirmation\'),scoring_signals():result2=score_part(out,b,\'confirmation\')\n        if result2[\'status\']!=\'complete\':raise RuntimeError(\'Caption confirmation đủ, metric chưa đủ; chạy ô 4 để chấm lại.\')\n        final_gate=gate(result2[\'metrics\']);atomic(out/\'confirmation_gate.json\',dict(final_gate,protocol=contract))\n        status=\'confirmation_passed\' if final_gate[\'passed\'] else \'confirmation_failed\'\n    else:status=\'development_failed\'\n    check_frozen();atomic(out/\'complete.json\',{\'protocol\':contract,\'status\':status,\'model_loads_this_process\':1,\'test_used\':False,\'improvement_proven\':False})\n    print(\'Pilot kết thúc: \'+status+\'. Chạy ô 4 xem bảng điểm và gửi ZIP.\',flush=True)\n\nif __name__==\'__main__\':\n    cfg=read(sys.argv[1]);out=Path(cfg[\'out\'])\n    try:run(cfg)\n    except BaseException as exc:\n        atomic(out/\'last_error.json\',{\'type\':type(exc).__name__,\'message\':str(exc),\'traceback\':traceback.format_exc()})\n        raise\n', 'scst_score.py': '"""CPU-only scorer and report. Missing results/metrics fail closed, never count as zero."""\nimport csv,html,os,signal,subprocess,sys\nfrom pathlib import Path\nfrom scst_support import *\nfrom scst_state import Checkpoints\nfrom caption_metrics import tokenize_checked\n\ndef verify_run(out,b,work):\n    validate_bundle(b);p=read(out/\'protocol.json\')\n    for key,expected in {\'config\':b[\'config\'],\'selection\':b[\'selection_ids\'],\n        \'checkpoint\':b[\'prior_protocol\'][\'checkpoint\'],\'vision\':b[\'prior_protocol\'][\'vision\'],\n        \'vision_hashes\':b[\'vision_hashes\'],\'source_sha\':b[\'source_sha\'],\n        \'precision\':b[\'prior_protocol\'][\'precision\'],\'decode\':b[\'prior_protocol\'][\'decode\'],\n        \'annotation_sha\':b[\'annotation_sha\'],\'question\':QUESTION,\'packages\':b[\'packages\'],\n        \'reward_data\':b[\'selection_ids\'][\'reward_rows\'],\'test_used\':False}.items():\n        if p.get(key)!=expected:raise ValueError(\'Protocol mismatch: \'+key)\n    if p[\'code\']!={f.name:sha(f) for f in sorted(work.glob(\'*.py\'))}:raise ValueError(\'Code changed; use the notebook that produced this run\')\n    if sha(out/\'reward.json\')!=p[\'reward_sha\']:raise ValueError(\'Reward identity changed\')\n    return p\n\ndef cache_meta(out,b,p,part,arm):\n    w=read(out/f\'weights_{arm}.json\');contract=identity(p)\n    if w[\'protocol\']!=contract:raise ValueError(\'Weights protocol mismatch\')\n    if arm==\'G\':\n        if w[\'weights\']!=identity(b[\'prior_protocol\'][\'checkpoint\']):raise ValueError(\'Wrong G weights\')\n    else:\n        meta=Checkpoints(out/\'checkpoints\'/arm,arm,contract).metadata()\n        if not meta or meta[\'step\']!=128 or w.get(\'step\')!=128 or meta[\'sha256\']!=w[\'weights\']:raise ValueError(\'Wrong final adapter\')\n    return {\'protocol\':contract,\'weights\':w[\'weights\'],\'selection\':b[\'selection_ids\'][part],\'arm\':arm,\'part\':part}\n\ndef meteor_process(source,target,log_path,timeout=600):\n    target.unlink(missing_ok=True)\n    with log_path.open(\'w\',encoding=\'utf-8\') as log:\n        proc=subprocess.Popen([sys.executable,str(Path(__file__).with_name(\'caption_metrics.py\')),\'--meteor\',str(source),str(target)],\n            stdout=log,stderr=subprocess.STDOUT,start_new_session=(os.name!=\'nt\'))\n        def stop():\n            if os.name==\'nt\':subprocess.run([\'taskkill\',\'/PID\',str(proc.pid),\'/T\',\'/F\'],capture_output=True)\n            else:\n                try:os.killpg(proc.pid,signal.SIGKILL)\n                except ProcessLookupError:pass\n            proc.wait()\n        try:\n            code=proc.wait(timeout=timeout)\n            return \'completed\' if code==0 and target.exists() else \'failed\'\n        except subprocess.TimeoutExpired:stop();return \'timeout\'\n        except BaseException:stop();raise\n\ndef calculate_metrics(folder,records,meteor_timeout=600):\n    from pycocoevalcap.bleu.bleu import Bleu\n    from pycocoevalcap.rouge.rouge import Rouge\n    from pycocoevalcap.cider.cider import Cider\n    folder=Path(folder);folder.mkdir(parents=True,exist_ok=True);metrics={};mi={}\n    for domain in [\'All\',\'Real\',\'Render\']:\n        selected=[r for r in records[\'G\'] if domain==\'All\' or r[\'subset\']==domain.lower()]\n        ids={r[\'id\'] for r in selected};refs=tokenize_checked({r[\'id\']:[{\'caption\':r[\'reference\']}] for r in selected});metrics[domain]={}\n        for arm in ARMS:\n            preds=tokenize_checked({r[\'id\']:[{\'caption\':r[\'prediction\']}] for r in records[arm] if r[\'id\'] in ids})\n            if set(refs)!=set(preds):raise ValueError(\'Unequal scored IDs\')\n            values={}\n            for scorer,names in [(Bleu(4),[\'BLEU-1\',\'BLEU-2\',\'BLEU-3\',\'BLEU-4\']),(Rouge(),[\'ROUGE-L\']),(Cider(),[\'CIDEr\'])]:\n                v,_=scorer.compute_score(refs,preds)\n                for k,s in zip(names,v if isinstance(v,list) else [v]):\n                    if not math.isfinite(float(s)):raise ValueError(\'Nonfinite metric\')\n                    values[k]=float(s)*100\n            metrics[domain][arm]=values;mi[domain+\'/\'+arm]={\'refs\':refs,\'preds\':preds}\n    atomic(folder/\'meteor_input.json\',mi)\n    status=meteor_process(folder/\'meteor_input.json\',folder/\'meteor_result.json\',folder/\'meteor.log\',timeout=meteor_timeout)\n    if status==\'completed\':\n        mr=read(folder/\'meteor_result.json\')\n        if set(mr)!=set(mi) or any(not math.isfinite(v) for v in mr.values()):raise ValueError(\'Invalid METEOR output\')\n        for key,v in mr.items():d,a=key.split(\'/\');metrics[d][a][\'METEOR\']=v\n    return metrics,\'complete\' if status==\'completed\' else \'partial_metrics\'\n\ndef score_part(out,b,part):\n    work=Path(__file__).parent;p=verify_run(out,b,work);folder=out/part;rows=b[part];records={}\n    counts={a:0 for a in ARMS}\n    for arm in ARMS:\n        if not (out/f\'weights_{arm}.json\').exists():continue\n        records[arm]=read_cache(folder,arm,cache_meta(out,b,p,part,arm),rows);counts[arm]=len(records[arm])\n    if any(n!=len(rows) for n in counts.values()):return {\'status\':\'incomplete\',\'counts\':counts,\'metrics\':{}}\n    sig={\'protocol\':identity(p),\'selection\':b[\'selection_ids\'][part],\n        \'captions\':{a:sha(folder/(a+\'.json\')) for a in ARMS},\'scorer\':\'pycocoevalcap==1.2, PTB; scale100\'}\n    report=folder/\'score_report.json\'\n    if report.exists():\n        old=read(report)\n        if old[\'identity\']!=sig:raise ValueError(\'Scored captions/identity changed\')\n        if old[\'status\']==\'complete\':\n            # Check that all required scores still exist and are finite.\n            if any(\'Thiếu \' in r for r in gate(old[\'metrics\'])[\'reasons\']):raise ValueError(\'Incomplete score marked complete\')\n            return old\n    metrics,status=calculate_metrics(folder,records)\n    result={\'status\':status,\'counts\':counts,\'identity\':sig,\'metrics\':metrics,\'gate\':gate(metrics)}\n    atomic(report,result);atomic(folder/\'metrics_x100.json\',metrics)\n    with (folder/\'caption_comparison.csv\').open(\'w\',encoding=\'utf-8-sig\',newline=\'\') as f:\n        w=csv.writer(f);w.writerow([\'id\',\'subset\',\'before\',\'after\',\'reference\',\'G\',\'A\',\'B\'])\n        for group in zip(*(records[a] for a in ARMS)):\n            r=group[0];w.writerow([r[\'id\'],r[\'subset\'],*r[\'image\'],r[\'reference\'],*[x[\'prediction\'] for x in group]])\n    return result\n\ndef completed_run(out,b,work):\n    if not (out/\'complete.json\').exists():return False\n    p=verify_run(out,b,work);end=read(out/\'complete.json\')\n    if end[\'protocol\']!=identity(p) or end[\'status\'] not in [\'development_failed\',\'confirmation_failed\',\'confirmation_passed\']:raise ValueError(\'Invalid completion marker\')\n    parts=[\'development\']+([] if end[\'status\']==\'development_failed\' else [\'confirmation\'])\n    for part in parts:\n        f=out/part;r=read(f/\'score_report.json\')\n        for a in ARMS:\n            records=read_cache(f,a,cache_meta(out,b,p,part,a),b[part])\n            if len(records)!=200 or sha(f/(a+\'.json\'))!=r[\'identity\'][\'captions\'][a]:raise ValueError(\'Completion caption mismatch\')\n        if r[\'identity\'][\'protocol\']!=identity(p) or r[\'status\']!=\'complete\':raise ValueError(\'Completion score mismatch\')\n        passed=gate(r[\'metrics\'])[\'passed\']\n        expected=(part==\'development\' and end[\'status\']!=\'development_failed\') or (part==\'confirmation\' and end[\'status\']==\'confirmation_passed\')\n        if passed!=expected:raise ValueError(\'Completion gate mismatch\')\n    return True\n\ndef write_report(out,results):\n    notes=\'G: gốc. A: LoRA học theo caption. B: LoRA học theo caption + SCST. Điểm ×100; B − G/B − A dương nghĩa là cao hơn.\'\n    caveat=\'Đây là pilot, chưa chứng minh tăng full test. Validation đã được dùng trong các lượt trước; hai phần ở lượt này không trùng ảnh theo đường dẫn. Gate đã khóa là điều kiện thực dụng, không phải kiểm định thống kê. A/B cùng số cặp và cập nhật; B tốn thêm tính toán. Không tự thay demo.\'\n    md=[\'# LoRA + SCST — kết quả pilot\',\'\',notes,\'\',caveat,\'\'];tables=[];csvrows=[]\n    for part,title in [(\'development\',\'Development: 200 cặp\'),(\'confirmation\',\'Confirmation: 200 cặp\')]:\n        result=results[part];md+=[\'## \'+title,\'\']\n        if result[\'status\']==\'incomplete\':\n            text=\'chưa đủ caption; \'+str(result.get(\'counts\',{}));md+=[text,\'\'];tables+=[\'<h2>\'+title+\'</h2><p>\'+html.escape(text)+\'</p>\'];continue\n        metrics=result[\'metrics\'];md+=[\'| Mô hình | \'+\' | \'.join(d+\' \'+k for d in [\'Real\',\'Render\',\'All\'] for k in METRICS)+\' |\',\'|---|\'+\'---:|\'*12]\n        table=\'<h2>\'+title+\'</h2><div style="overflow-x:auto"><table><thead><tr><th rowspan="2">Mô hình</th>\'+\'\'.join(\'<th colspan="4">\'+d+\'</th>\' for d in [\'Real\',\'Render\',\'All\'])+\'</tr><tr>\'+\'\'.join(\'<th>\'+k+\'</th>\' for _ in range(3) for k in METRICS)+\'</tr></thead><tbody>\'\n        for a in [\'G\',\'A\',\'B\',\'B − G\',\'B − A\']:\n            values=[]\n            for d in [\'Real\',\'Render\',\'All\']:\n                for k in METRICS:\n                    if a in ARMS:v=metrics[d][a].get(k)\n                    else:\n                        x=metrics[d][\'B\'].get(k);y=metrics[d][a[-1]].get(k);v=None if x is None or y is None else x-y\n                    values.append(\'chưa có\' if v is None else format(v,\'+.4f\' if a not in ARMS else \'.4f\'))\n            md+=[\'| \'+a+\' | \'+\' | \'.join(values)+\' |\'];csvrows.append([part,a]+values)\n            table+=\'<tr><td>\'+a+\'</td>\'+\'\'.join(\'<td>\'+v+\'</td>\' for v in values)+\'</tr>\'\n        decision=gate(metrics);statement=\'Đạt ngưỡng pilot.\' if decision[\'passed\'] else \'Chưa đạt: \'+\'; \'.join(decision[\'reasons\'])\n        md+=[\'\',statement,\'\'];tables+=[table+\'</tbody></table></div><p>\'+html.escape(statement)+\'</p>\']\n    for name in [\'eta\',\'budget\',\'last_error\']:\n        if (out/(name+\'.json\')).exists():\n            data=read(out/(name+\'.json\'))\n            if name==\'last_error\':data={k:data[k] for k in [\'type\',\'message\']}\n            if name==\'budget\':data={k:data[k] for k in [\'spent\',\'limit\',\'active\']}\n            md+=[\'**\'+name+\'**: \'+json.dumps(data,ensure_ascii=False),\'\']\n    (out/\'KET_QUA.md\').write_text(\'\\n\'.join(md),encoding=\'utf-8\')\n    page=\'<!doctype html><html lang="vi"><meta charset="utf-8"><title>LoRA + SCST</title><style>body{font:15px Arial;margin:28px;color:#17212b}p{line-height:1.5}table{border-collapse:collapse;width:100%;font-size:13px}td,th{padding:10px;border-bottom:1px solid #ddd;text-align:right}td:first-child{white-space:nowrap;text-align:left}thead{border-block:2px solid #333}h2{margin-top:32px}</style><body><h1>LoRA + SCST — pilot</h1><p>\'+html.escape(notes)+\'</p><p>\'+html.escape(caveat)+\'</p>\'+\'\'.join(tables)+\'</body></html>\'\n    (out/\'BANG_SO_SANH.html\').write_text(page,encoding=\'utf-8\')\n    with (out/\'bang_diem.csv\').open(\'w\',encoding=\'utf-8-sig\',newline=\'\') as f:\n        w=csv.writer(f);w.writerow([\'part\',\'model\']+[d+\' \'+k for d in [\'Real\',\'Render\',\'All\'] for k in METRICS]);w.writerows(csvrows)\n\ndef report(out,b):\n    results={}\n    for part in [\'development\',\'confirmation\']:\n        if not (out/\'protocol.json\').exists():results[part]={\'status\':\'incomplete\',\'counts\':{a:0 for a in ARMS}}\n        else:results[part]=score_part(out,b,part)\n    write_report(out,results)\n    atomic(out/\'summary.json\',{\'results\':results,\'test_used\':False,\'improvement_proven\':False})\n    dev=results[\'development\'];confirm=results[\'confirmation\']\n    if dev[\'status\']==\'complete\':\n        p=read(out/\'protocol.json\');dg=gate(dev[\'metrics\']);atomic(out/\'development_gate.json\',dict(dg,protocol=identity(p)))\n        status=None\n        if not dg[\'passed\']:status=\'development_failed\'\n        elif confirm[\'status\']==\'complete\':\n            cg=gate(confirm[\'metrics\']);atomic(out/\'confirmation_gate.json\',dict(cg,protocol=identity(p)))\n            status=\'confirmation_passed\' if cg[\'passed\'] else \'confirmation_failed\'\n        if status:atomic(out/\'complete.json\',{\'protocol\':identity(p),\'status\':status,\'test_used\':False,\'improvement_proven\':False})\n    print(\'Đã xuất bảng Real / Render / All. Chưa chạy full test.\',flush=True)\n\nif __name__==\'__main__\':\n    def interrupted(signum,frame):raise InterruptedError(\'Scoring interrupted; captions preserved\')\n    signal.signal(signal.SIGTERM,interrupted)\n    report(Path(sys.argv[1]),read(Path(__file__).with_name(\'bundle.json\')))\n', 'caption_metrics.py': 'import json, os, subprocess, sys, tempfile\nfrom pathlib import Path\nfrom scst_support import atomic\n\ndef java_options():\n    return ([\'-XX:-UseContainerSupport\'] if sys.platform.startswith(\'linux\') else []) + [\'-Dfile.encoding=UTF-8\']\n\ndef tokenize_checked(captions):\n    """Same Stanford PTB rules, with UTF-8 and a checked one-line-per-caption protocol."""\n    from pycocoevalcap.tokenizer import ptbtokenizer\n    flattened = [(key, item[\'caption\']) for key, items in captions.items() for item in items]\n    if not flattened:\n        raise ValueError(\'No captions to tokenize\')\n    jar = Path(ptbtokenizer.__file__).resolve().parent / ptbtokenizer.STANFORD_CORENLP_3_4_1_JAR\n    with tempfile.TemporaryDirectory(prefix=\'m3diff_ptb_\') as folder:\n        source = Path(folder) / \'captions.txt\'\n        source.write_text(\'\\n\'.join((s.replace(\'\\n\', \' \').replace(\'\\r\', \' \') for _, s in flattened)) + \'\\n\', encoding=\'utf-8\')\n        result = subprocess.run([\'java\'] + java_options() + [\'-cp\', str(jar), \'edu.stanford.nlp.process.PTBTokenizer\', \'-preserveLines\', \'-lowerCase\', \'-encoding\', \'UTF-8\', str(source)], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, encoding=\'utf-8\', check=True, timeout=60)\n    lines = result.stdout.split(\'\\n\')\n    if lines and lines[-1] == \'\':\n        lines.pop()\n    if len(lines) != len(flattened) or any((\'[warning]\' in line or \'[os,container]\' in line for line in lines)):\n        raise ValueError(\'Invalid PTB output: unexpected diagnostic text or caption count; scores not produced\')\n    tokenized = {key: [] for key in captions}\n    for (key, _), line in zip(flattened, lines):\n        tokenized[key].append(\' \'.join((w for w in line.rstrip().split(\' \') if w not in ptbtokenizer.PUNCTUATIONS)))\n    return tokenized\n\ndef meteor_only(source, target):\n    from pycocoevalcap.meteor.meteor import Meteor\n    os.environ[\'JAVA_TOOL_OPTIONS\'] = (os.environ.get(\'JAVA_TOOL_OPTIONS\', \'\') + \' \' + \' \'.join(java_options())).strip()\n    data = json.loads(Path(source).read_text(encoding=\'utf-8\'))\n    scorer = Meteor()\n    scores = {}\n    try:\n        for key, item in data.items():\n            clean = lambda d: {i: [\' \'.join(s.replace(\'|\', \' \').split()) for s in captions] for i, captions in d.items()}\n            value, _ = scorer.compute_score(clean(item[\'refs\']), clean(item[\'preds\']))\n            scores[key] = float(value) * 100\n        atomic(target, scores)\n    finally:\n        scorer.meteor_p.kill()\n        scorer.meteor_p.wait()\n        for stream in [scorer.meteor_p.stdin, scorer.meteor_p.stdout, scorer.meteor_p.stderr]:\n            if stream is not None:\n                stream.close()\n        if scorer.lock.locked():\n            scorer.lock.release()\n\nif __name__=="__main__":\n    meteor_only(sys.argv[2],sys.argv[3])\n', 'inference_tokens.py': "def token_layout(tokenizer, row, answer=True):\n    start = tokenizer.convert_tokens_to_ids('<|im_start|>')\n    end = tokenizer.convert_tokens_to_ids('<|im_end|>')\n    if start == end or start == tokenizer.unk_token_id or end == tokenizer.unk_token_id:\n        raise ValueError('Wrong Qwen tokenizer')\n    enc = lambda s: tokenizer.encode(s, add_special_tokens=False)\n    ids = [start] + enc('system\\nYou are a helpful assistant.') + [end] + enc('\\n')\n    labels = [-100] * len(ids)\n    for turn in row['conversations']:\n        role = {'human': 'user', 'gpt': 'assistant'}[turn['from']]\n        if not answer and role != 'user':\n            continue\n        header = [start] + enc(role + '\\n')\n        body = []\n        for i, chunk in enumerate(turn['value'].split('<image>')):\n            if i:\n                body.append(-200)\n            body.extend(enc(chunk))\n        segment = header + body + [end] + enc('\\n')\n        ids += segment\n        labels += [-100] * len(segment) if role == 'user' else [-100] * len(header) + body + [end] + enc('\\n')\n    if not answer:\n        tail = [start] + enc('assistant\\n')\n        ids += tail\n        labels += [-100] * len(tail)\n    if ids.count(-200) != 2 or any((y != -100 for x, y in zip(ids, labels) if x == -200)):\n        raise ValueError('Image token masking')\n    if answer and (not any((y != -100 for y in labels))):\n        raise ValueError('No answer labels')\n    if len(ids) + 1458 > 4096:\n        raise ValueError('Sequence exceeds 4096; no silent truncation')\n    return (ids, labels)\n"}
BUNDLE_GZIP = '...'
SOURCE_SHA = '8a0c3e930942dcc2ea0d6fa69fc9a7226d47362e127583313b350e8d2961dffa'
SOURCE_ZIP_BASE64 = '...'
for name,code in EMBEDDED_SOURCES.items():
    compile(code,name,'exec');(WORK/name).write_text(code,encoding='utf-8')
bundle=json.loads(gzip.decompress(base64.b64decode(BUNDLE_GZIP)))
(WORK/'bundle.json').write_text(json.dumps(bundle,ensure_ascii=False,indent=2),encoding='utf-8')
archive=WORK/'source.zip';archive.write_bytes(base64.b64decode(SOURCE_ZIP_BASE64))
if hashlib.sha256(archive.read_bytes()).hexdigest()!=SOURCE_SHA:raise ValueError('Source zip checksum')
with zipfile.ZipFile(archive) as z:
    for name in z.namelist():
        if not (SOURCE/name).resolve().is_relative_to(SOURCE.resolve()):raise ValueError('Unsafe source archive')
    z.extractall(SOURCE)
ENV=dict(os.environ,PYTHONPATH=str(WORK)+os.pathsep+str(SOURCE),PYTHONUNBUFFERED='1',TOKENIZERS_PARALLELISM='false')

def run_job(command,log_name,timeout=1800):
    # A child process keeps torch off the Colab kernel and releases GPU when done.
    log_path=OUT/log_name
    with log_path.open('a',encoding='utf-8') as log:
        proc=subprocess.Popen(command,cwd=WORK,env=ENV,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
            text=True,encoding='utf-8',errors='replace',bufsize=1,start_new_session=True)
        def forward():
            for line in proc.stdout:
                print(line,end='',flush=True);log.write(line);log.flush()
        reader=threading.Thread(target=forward,daemon=True);reader.start()
        try:code=proc.wait(timeout=timeout)
        except BaseException:
            # Let the scorer stop its own METEOR/JVM group before the hard-stop fallback.
            try:os.killpg(proc.pid,signal.SIGTERM)
            except ProcessLookupError:pass
            try:proc.wait(timeout=120)
            except subprocess.TimeoutExpired:
                try:os.killpg(proc.pid,signal.SIGKILL)
                except ProcessLookupError:pass
                proc.wait()
            reader.join(timeout=10)
            print('Đã dừng tiến trình. Checkpoint/caption đã lưu vẫn ở Drive. Chạy ô 4 lấy báo cáo; mất phiên thì chạy lại 1→2→3.')
            raise
        reader.join(timeout=10)
    if code:raise RuntimeError(f'Bước chưa xong (mã {code}). Xem log {log_path.name}; chạy ô 4 vẫn lấy được ZIP.')

if importlib.util.find_spec('uv') is None:run_job([sys.executable,'-m','pip','install','-q','uv'],'setup.log')
UV=[sys.executable,'-m','uv']
if not PY.is_file():
    run_job(UV+['python','install','3.10'],'setup.log')
    run_job(UV+['venv',str(VENV),'--python','3.10'],'setup.log')
run_job([str(PY),'-c','import sys; assert sys.version_info[:2]==(3,10); print("Python riêng:",sys.version.split()[0])'],'setup.log')
run_job(UV+['pip','install','--python',str(PY),'huggingface-hub==0.36.2','Pillow==11.3.0','numpy==1.26.4'],'setup.log')
print('Ô 1 hoàn tất. Lần đầu: 2 → 3 → 4. Chỉ muốn chấm/tải báo cáo: 1 → 4 (dùng CPU cũng được).')


Mounted at /content/drive
Installed Python 3.10.22 in 857ms
 + cpython-3.10.22-linux-x86_64-gnu (python3.10)
Using CPython 3.10.22
Creating virtual environment at: /content/m3diff-train-venv
Activate with: source /content/m3diff-train-venv/bin/activate
Python riêng: 3.10.22
Using Python 3.10.22 environment at: /content/m3diff-train-venv
Resolved 15 packages in 174ms
Prepared 15 packages in 303ms
Installed 15 packages in 20ms
 + certifi==2026.10.23
 + charset-normalizer==3.5.2
 + filelock==4.1.0
 + fsspec==2026.9.0
 + hf-xet==1.8.0
 + huggingface-hub==0.36.2
 + idna==3.20
 + numpy==1.26.4
 + packaging==26.3
 + pillow==11.3.0
 + pyyaml==6.0.3
 + requests==2.34.2
 + tqdm==4.70.1
 + typing-extensions==4.16.0
 + urllib3==2.8.0
Ô 1 hoàn tất. Lần đầu: 2 → 3 → 4. Chỉ muốn chấm/tải báo cáo: 1 → 4 (dùng CPU cũng được).


In [ ]:
#@title 2. Chuẩn bị A100 80GB, trọng số gốc, ảnh và reward — dùng lại dữ liệu đã có
if 'OUT' not in globals():raise RuntimeError('Chạy ô 1 trước.')
if (OUT/'complete.json').exists():
    print('Đã có kết quả cuối. Chạy ô 4 để kiểm tra và tải báo cáo; không dùng thêm GPU.')
else:
    if not shutil.which('nvidia-smi'):raise RuntimeError('Chọn A100 80GB: Runtime → Change runtime type.')
    gpu=subprocess.check_output(['nvidia-smi','--query-gpu=name,memory.total,memory.used','--format=csv,noheader,nounits'],text=True).strip()
    print('GPU / tổng MB / đang dùng MB:',gpu)
    if len(gpu.splitlines())!=1:raise RuntimeError('Cần đúng một GPU A100 80GB.')
    name,total,used=[x.strip() for x in gpu.split(',')]
    if 'A100' not in name or int(total)<75000 or int(used)>2500:raise RuntimeError('Cần A100 80GB trống. Tắt tiến trình model/demo khác trước; chưa tải dữ liệu lớn.')
    EXPECTED=dict(bundle['packages'], scipy='1.11.4', setuptools='69.5.1') # Phụ thuộc runtime của bitsandbytes và Triton
    probe = """import json, sys
from importlib import metadata
result = {}
for name in json.loads(sys.argv[1]):
    try:
        result[name] = metadata.version(name)
    except metadata.PackageNotFoundError:
        result[name] = None
print(json.dumps(result))
"""
    versions=json.loads(subprocess.check_output([str(PY),'-c',probe,json.dumps(list(EXPECTED))],text=True))
    if any(versions[n]!=EXPECTED[n] for n in ['torch','torchvision']):
        run_job(UV+['pip','install','--python',str(PY),'torch==2.2.2','torchvision==0.17.2','--index-url','https://download.pytorch.org/whl/cu121'],'setup.log')
    deps=[n+'=='+v for n,v in EXPECTED.items() if n not in ['torch','torchvision'] and versions[n]!=v]
    if deps:run_job(UV+['pip','install','--python',str(PY)]+deps,'setup.log')
    preflight = "import setuptools\nimport torch\nfrom PIL import Image\nfrom peft import LoraConfig, get_peft_model, get_peft_model_state_dict, set_peft_model_state_dict\nfrom transformers import AutoTokenizer, GenerationConfig\nfrom llava.model.language_model.llava_qwen import LlavaQwenForCausalLM, LlavaQwenConfig\nfrom llava.mm_utils import process_images\nfrom caption_metrics import tokenize_checked\nfrom scst_reward import FrozenCider\nfrom scst_score import score_part, verify_run, cache_meta, completed_run\nimport scipy, bitsandbytes, torchvision\nfrom scipy.stats import norm\nfrom importlib.metadata import version\nimport scst_runner\nassert version('setuptools') == '69.5.1', 'Wrong setuptools version'\nassert version('scipy') == '1.11.4', 'Wrong scipy version'\nassert abs(norm.cdf(0) - 0.5) < 1e-12\nprint('Kiểm tra import OK: SciPy, bitsandbytes, PEFT, M3Diff và SCST. Chưa nạp trọng số.')\n"
    run_job([str(PY),'-c',preflight],'setup.log',timeout=180)
    if not shutil.which('java'):
        run_job(['apt-get','-qq','update'],'setup.log',timeout=300)
        run_job(['apt-get','-qq','install','-y','default-jre-headless'],'setup.log',timeout=300)
    CONFIG={'work':str(WORK),'out':str(OUT),'storage':str(STORAGE),'source_sha':SOURCE_SHA,
        'model':'/content/m3diff_train_checkpoint','images':'/content/m3diff_scst_images'}
    (WORK/'prepare_config.json').write_text(json.dumps(CONFIG),encoding='utf-8')
    run_job([str(PY),str(WORK/'scst_prepare.py'),str(WORK/'prepare_config.json')],'prepare.log',timeout=7200)


GPU / tổng MB / đang dùng MB: NVIDIA A100-SXM4-80GB, 81920, 0
Using Python 3.10.22 environment at: /content/m3diff-train-venv
Resolved 25 packages in 1.79s
Prepared 20 packages in 20.73s
Installed 20 packages in 175ms
 + jinja2==3.1.6
 + markupsafe==3.0.3
 + mpmath==1.3.0
 + networkx==3.4.2
 + nvidia-cublas-cu12==12.1.3.1
 + nvidia-cuda-cupti-cu12==12.1.105
 + nvidia-cuda-nvrtc-cu12==12.1.105
 + nvidia-cuda-runtime-cu12==12.1.105
 + nvidia-cudnn-cu12==8.9.2.26
 + nvidia-cufft-cu12==11.0.2.54
 + nvidia-curand-cu12==10.3.2.106
 + nvidia-cusolver-cu12==11.4.5.107
 + nvidia-cusparse-cu12==12.1.0.106
 + nvidia-nccl-cu12==2.19.3
 + nvidia-nvjitlink-cu12==12.9.86
 + nvidia-nvtx-cu12==12.1.105
 + sympy==1.14.0
 + torch==2.2.2+cu121
 + torchvision==0.17.2+cu121
 + triton==2.2.0
Using Python 3.10.22 environment at: /content/m3diff-train-venv
Resolved 52 packages in 229ms
Prepared 17 packages in 1.97s
Installed 17 packages in 41ms
 + accelerate==0.29.1
 + bitsandbytes==0.41.0
 + einops==0.6.1
 + 

In [ ]:
#@title 3. Kiểm tra nhanh → G/A/B → xác nhận nếu đạt — tự lưu tiếp

if 'OUT' not in globals():raise RuntimeError('Chạy ô 1 trước.')
if (OUT/'complete.json').exists():
    print('Đã có kết quả cuối; chạy ô 4 kiểm tra và tải báo cáo, không train lại.')
else:
    if not (WORK/'prepared.json').is_file():raise RuntimeError('Chạy ô 2 trước.')
    spent=json.loads((OUT/'budget.json').read_text(encoding='utf-8'))['spent'] if (OUT/'budget.json').exists() else 0
    remaining=10800-spent
    if remaining<150:raise RuntimeError('Hết phần ngân sách đủ an toàn cho pilot. Chạy ô 4; không tự xóa thư mục để chạy lại.')
    print('G = gốc | A = LoRA thường | B = LoRA + SCST. Smoke được bỏ trước pilot.')
    print('Trần còn lại:',round(remaining/60,1),'phút, không phải ETA. ETA được đo sau smoke. Ô này tự chạy hai nhánh; không cần bấm thêm.')
    run_job([str(PY),str(WORK/'scst_runner.py'),str(WORK/'prepared.json')],'training.log',timeout=remaining-120)


G = gốc | A = LoRA thường | B = LoRA + SCST. Smoke được bỏ trước pilot.
Trần còn lại: 97.1 phút, không phải ETA. ETA được đo sau smoke. Ô này tự chạy hai nhánh; không cần bấm thêm.
Nạp một model vào GPU. G/A/B dùng chung model này và đổi adapter.
The checkpoint contains an unfrozen vision tower; loading it eagerly.
/content/m3diff-train-venv/lib/python3.10/site-packages/torch/nn/modules/module.py:2025: UserWarning: for vision_model.embeddings.patch_embedding.weight: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=True` to assign items in the state dictionary to their corresponding key in the module instead of copying them in place?)
  warnings.warn(f'for {key}: copying from a non-meta parameter in the checkpoint to a meta '
/content/m3diff-train-venv/lib/python3.10/site-packages/torch/nn/modules/module.py:2025: UserWarning: for vision_model.embeddings.patch_embedding.bias: copying from a non-m

In [ ]:
#@title 4. Xem điểm Real / Render / All và tải ZIP — chạy cả khi lỗi hoặc chưa đủ
if 'OUT' not in globals():raise RuntimeError('Chạy ô 1 trước.')
from google.colab import files
from IPython.display import display,HTML
try:
    if not shutil.which('java'):
        run_job(['apt-get','-qq','update'],'score_setup.log',timeout=300)
        run_job(['apt-get','-qq','install','-y','default-jre-headless'],'score_setup.log',timeout=300)
    run_job(UV+['pip','install','--python',str(PY),'pycocoevalcap==1.2'],'score_setup.log',timeout=300)
    run_job([str(PY),str(WORK/'scst_score.py'),str(OUT)],'scoring.log',timeout=1800)
    display(HTML((OUT/'BANG_SO_SANH.html').read_text(encoding='utf-8')))
except Exception as exc:
    (OUT/'score_error.txt').write_text(str(exc),encoding='utf-8')
    print('Chưa chấm xong:',exc,'— vẫn tải ZIP. Không cần sinh lại caption đã lưu.')
archive=Path('/content/M3Diff_LoRA_SCST_bao_cao.zip')
with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.rglob('*')):
        if p.is_file() and p.suffix in ['.json','.jsonl','.csv','.md','.html','.txt','.log']:
            z.write(p,p.relative_to(OUT))
    for name in EMBEDDED_SOURCES:z.write(WORK/name,'code/'+name)
    z.write(WORK/'bundle.json','code/bundle.json')
files.download(str(archive))
print('Gửi lại M3Diff_LoRA_SCST_bao_cao.zip. Trọng số/optimizer nằm trên Drive, không nhét vào ZIP.')
print('Drive:',OUT)
print('Xong việc nhớ Disconnect and delete runtime để ngừng dùng GPU. Demo vẫn dùng gốc.')


Using Python 3.10.22 environment at: /content/m3diff-train-venv
Checked 1 package in 1ms
Đã xuất bảng Real / Render / All. Chưa chạy full test.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Gửi lại M3Diff_LoRA_SCST_bao_cao.zip. Trọng số/optimizer nằm trên Drive, không nhét vào ZIP.
Drive: /content/drive/MyDrive/M3Diff_Storage/MDP_Projector_v1/experiments/scst_lora_20261031
Xong việc nhớ Disconnect and delete runtime để ngừng dùng GPU. Demo vẫn dùng gốc.
